# RLSP Reproduction — Qwen2.5-1.5B — Aime - Lightning.ai L4


In [ ]:
!nvidia-smi

## 1. Setup & Clone

In [ ]:
print("Hello World")

In [ ]:
import os
print("Staring")
os.chdir('/teamspace/studios/this_studio')
!rm -rf /teamspace/studios/this_studio/Emergence-of-Thinking
!git clone https://github.com/GuanghaoYe/Emergence-of-Thinking.git
os.chdir('/teamspace/studios/this_studio/Emergence-of-Thinking')

print('---> Fixing Ray version...')
for root, dirs, files in os.walk('.'):
    if '.git' in root:
        continue
    for file in files:
        if file.endswith(('.txt', '.py', '.toml')):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r') as f:
                    content = f.read()
                if 'ray' in content.lower() and '2.12.0' in content:
                    with open(filepath, 'w') as f:
                        f.write(content.replace('2.12.0', '2.31.0'))
                    print(f'  Fixed: {filepath}')
            except:
                pass

print('\n---> 0/2: Installing PyTorch...')
!pip install torch torchvision torchaudio

print('\n---> 1/2: vLLM, Ray, Deepspeed, PEFT, Flask...')
!pip install vllm==0.6.3 deepspeed peft wandb bitsandbytes "ray[default]" flask

print('\n---> 2/2: flash-attn...')
!pip install flash-attn --no-cache-dir --no-build-isolation || echo 'flash-attn failed, continuing'

In [ ]:
import os
print("Staring")
os.chdir('/teamspace/studios/this_studio')
!rm -rf /teamspace/studios/this_studio/Emergence-of-Thinking
!git clone https://github.com/GuanghaoYe/Emergence-of-Thinking.git
os.chdir('/teamspace/studios/this_studio/Emergence-of-Thinking')

print('---> Fixing Ray version...')
for root, dirs, files in os.walk('.'):
    if '.git' in root:
        continue
    for file in files:
        if file.endswith(('.txt', '.py', '.toml')):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r') as f:
                    content = f.read()
                if 'ray' in content.lower() and '2.12.0' in content:
                    with open(filepath, 'w') as f:
                        f.write(content.replace('2.12.0', '2.31.0'))
                    print(f'  Fixed: {filepath}')
            except:
                pass

print('\n---> 0/3: Installing PyTorch...')
!pip install torch torchvision torchaudio

print('\n---> 1/3: Core project...')
!pip install -e . --no-build-isolation

print('\n---> 2/3: vLLM, Ray, Deepspeed, PEFT, Flask...')
!pip install vllm==0.6.3 deepspeed peft wandb bitsandbytes "ray[default]" flask

print('\n---> 3/3: flash-attn...')
!pip install flash-attn --no-cache-dir --no-build-isolation || echo 'flash-attn failed, continuing'

print('Setup complete!')

## 2. Transformers fix

In [ ]:
!pip install transformers==4.45.2 accelerate -q

## 3. Download Qwen2.5-1.5B-Instruct

In the following code cell you should attach your Hugging Face token

In [ ]:
import os
# Attach your Hugging Face token
os.environ["HF_TOKEN"] = "....."

model_path = '/teamspace/studios/this_studio/models/Qwen2.5-1.5B-Instruct'
if not os.path.exists(model_path):
    !pip install huggingface_hub --quiet
    !huggingface-cli download Qwen/Qwen2.5-1.5B-Instruct \
        --local-dir /teamspace/studios/this_studio/models/Qwen2.5-1.5B-Instruct
    print('Model downloaded')
else:
    print('Model already present, skipping')

In [ ]:
!pip install latex2sympy2

## 4. Dataset & Directories

In [ ]:
import os, json

data_path = '/teamspace/studios/this_studio/Emergence-of-Thinking/data/aime_formatted_qwen.jsonl'
print('Training data exists:', os.path.exists(data_path))

with open(data_path) as f:
    sample = json.loads(f.readline())
print('Sample keys:', list(sample.keys()))
print('Sample input:', sample.get('input', '')[:200])

eval_data = '/teamspace/studios/this_studio/Emergence-of-Thinking/evaluation/data/AIME'
print('Eval data exists:', os.path.exists(eval_data))

os.makedirs('/tmp/code', exist_ok=True)
os.makedirs('/teamspace/studios/this_studio/Emergence-of-Thinking/logs/openrlhf_train_ppo', exist_ok=True)
print('Directories ready')

with open('/teamspace/studios/this_studio/Emergence-of-Thinking/data/aime_formatted_qwen.jsonl') as f_in, \
     open('/teamspace/studios/this_studio/Emergence-of-Thinking/data/aime_qwen.jsonl', 'w') as f_out:
    for line in f_in:
        item = json.loads(line)
        item['input'] = f"<|im_start|>user\nProblem: {item['problem']}<|im_end|>\n<|im_start|>assistant\n"
        f_out.write(json.dumps(item) + '\n')
print("Dataset OK!")

## 5. Clean old checkpoints

In [ ]:
import shutil, os

shutil.rmtree('/teamspace/studios/this_studio/checkpoints', ignore_errors=True)
shutil.rmtree('/tmp/ray', ignore_errors=True)

total, used, free = shutil.disk_usage('/teamspace/studios/this_studio')
print(f'Persistent storage -- Free: {free/1e9:.1f} GB')

total, used, free = shutil.disk_usage('/tmp')
print(f'/tmp -- Free: {free/1e9:.1f} GB')
print('Ready for training')


## 6. Patches

In [ ]:
# Patch 1: DeepSpeed lm_head bug
filepath = '/teamspace/studios/this_studio/Emergence-of-Thinking/openrlhf/utils/deepspeed/deepspeed.py'
with open(filepath, 'r') as f:
    lines = f.readlines()

patched = False
new_lines = []
for line in lines:
    if 'assert state_dict_keys.issubset(' in line and not patched:
        indent = len(line) - len(line.lstrip())
        new_lines.append(' ' * indent + 'state_dict_keys -= {"base_model.model.lm_head.weight"}\n')
        patched = True
    new_lines.append(line)

if patched:
    with open(filepath, 'w') as f:
        f.writelines(new_lines)
    print('DeepSpeed patch applied')
else:
    print('Already patched or pattern not found')

# Patch 2: Ray GPU override
import ray._private.resource_and_label_spec as _spec
import inspect, pathlib

fpath = pathlib.Path(inspect.getfile(_spec))
src = fpath.read_text()

old = """            raise ValueError(
                f"Attempting to start raylet with {num_accelerators} "
                f"{accelerator_resource_name}, "
                f"{accelerator_manager.get_visible_accelerator_ids_env_var()} "
                f"contains {visible_accelerator_ids}."
            )"""

new = """            pass  # patched: allow virtual GPU override"""

if old in src:
    fpath.write_text(src.replace(old, new))
    print('Ray patch applied')
else:
    print('Ray patch: already applied or pattern mismatch')

## 7. PPO training script



In [ ]:
import os
script_content = r"""#!/bin/bash
set -x
ray stop --force 2>/dev/null || true
sleep 3
CUDA_VISIBLE_DEVICES=0 \
RAY_EXPERIMENTAL_NOSET_CUDA_VISIBLE_DEVICES=1 \
RAY_DISABLE_GPU_USAGE_STATS=1 \
ray start --head --num-gpus 2 \
    --dashboard-host=0.0.0.0 \
    --include-dashboard=True \
    --disable-usage-stats \
    --object-store-memory=2000000000
echo "Waiting 15 seconds..."
sleep 15s
ray job submit --address="http://127.0.0.1:8265" \
    --runtime-env-json='{"working_dir": "/teamspace/studios/this_studio/Emergence-of-Thinking"}' \
    -- python -m openrlhf.cli.train_ppo_ray \
    --ref_num_nodes 1 \
    --ref_num_gpus_per_node 1 \
    --reward_num_nodes 1 \
    --reward_num_gpus_per_node 1 \
    --critic_num_nodes 1 \
    --critic_num_gpus_per_node 1 \
    --actor_num_nodes 1 \
    --actor_num_gpus_per_node 1 \
    --vllm_num_engines 0 \
    --vllm_tensor_parallel_size 1 \
    --colocate_actor_ref \
    --colocate_critic_reward \
    --pretrain /teamspace/studios/this_studio/models/Qwen2.5-1.5B-Instruct \
    --save_path /teamspace/studios/this_studio/checkpoints \
    --micro_train_batch_size 2 \
    --train_batch_size 8 \
    --micro_rollout_batch_size 4 \
    --rollout_batch_size 32 \
    --max_samples 480 \
    --max_epochs 1 \
    --num_episodes 20 \
    --prompt_max_len 512 \
    --generate_max_len 512 \
    --zero_stage 2 \
    --bf16 \
    --actor_learning_rate 2e-7 \
    --critic_learning_rate 2e-6 \
    --init_kl_coef 0.01 \
    --lambd 0.95 \
    --save_steps 9999 \
    --save_hf_ckpt \
    --disable_ds_ckpt \
    --prompt_data /teamspace/studios/this_studio/Emergence-of-Thinking/data/aime_qwen.jsonl \
    --input_key input \
    --normalize_reward \
    --gradient_checkpointing \
    --remote_rm_url http://127.0.0.1:8000/get_reward \
    --temperature 0.7 \
    --max_ckpt_num 5 \
    --lora_rank 64 \
    --lora_alpha 128 \
    --target_modules q_proj k_proj v_proj o_proj
"""
script_path = '/teamspace/studios/this_studio/Emergence-of-Thinking/train_ppo_1gpu_1b5.sh'
with open(script_path, 'w') as f:
    f.write(script_content)
os.chmod(script_path, 0o755)
print('Script written')

## 8. Start ORM Server

In [ ]:
import subprocess, time, urllib.request, json as _json, sys
import json

!pkill -f orm_server_efficient || true
time.sleep(2)

log_file = open('/teamspace/studios/this_studio/orm_server.log', 'w')
orm_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'openrlhf.cli.orm_server_efficient',
        '--dataset',        'evaluation/data/aime',
        '--model_name',     '/teamspace/studios/this_studio/models/Qwen2.5-1.5B-Instruct',
        '--log_dir',        './logs/openrlhf_train_ppo',
        '--length_penalty', '20',
        '--use_gpt',        '0',
    ],
    cwd='/teamspace/studios/this_studio/Emergence-of-Thinking',
    stdout=log_file,
    stderr=subprocess.STDOUT
)
print(f'ORM server PID: {orm_proc.pid}')
time.sleep(15)
try:
    dummy = _json.dumps({'query': ['test \\boxed{42}']}).encode()
    req = urllib.request.Request('http://127.0.0.1:8000/get_reward', data=dummy,
        headers={'Content-Type': 'application/json'}, method='POST')
    resp = urllib.request.urlopen(req, timeout=10)
    print(f'Server UP: {resp.read().decode()[:100]}')
except Exception as e:
    print(f'Error: {e}')
    

dummy = json.dumps({'query': ['test input \\boxed{42}']}).encode()
req = urllib.request.Request(
    'http://127.0.0.1:8000/get_reward',
    data=dummy,
    headers={'Content-Type': 'application/json'},
    method='POST'
)
try:
    resp = urllib.request.urlopen(req, timeout=5)
    print('Server UP:', resp.read().decode())
except Exception as e:
    print('Server DOWN:', e)


In [ ]:
import pathlib

fpath = pathlib.Path("/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ray/_private/worker.py")
src = fpath.read_text()

old = """        if self.original_visible_accelerator_ids.get(resource_name, None) is not None:
            original_ids = self.original_visible_accelerator_ids[resource_name]
            assigned_ids = {str(original_ids[i]) for i in assigned_ids}"""

new = """        if self.original_visible_accelerator_ids.get(resource_name, None) is not None:
            original_ids = self.original_visible_accelerator_ids[resource_name]
            # patched: clamp to available ids for virtual GPU trick
            assigned_ids = {str(original_ids[i]) if i < len(original_ids) else str(original_ids[0]) for i in assigned_ids}"""

if old in src:
    fpath.write_text(src.replace(old, new))
    print(" worker.py patch implemented!")
else:
    print("Pattern not found")

## 9. Run PPO Training
This runs for approximately 22 hours on one L4 GPU.

In [ ]:
import os
os.makedirs('/tmp/code', exist_ok=True)

print('Starting PPO training...')
print('Logs in real-time below\n')

!cd /teamspace/studios/this_studio/Emergence-of-Thinking && bash train_ppo_1gpu_1b5.sh
